# Lab 01 — Eval sobre un Agente Gatekeeper

En este lab construimos y evaluamos un **agente Claude** que actúa como guardián SDLC.

## Arquitectura del eval

```
┌─────────────────────────────────────────────────────────────┐
│                        EVAL LOOP                            │
│                                                             │
│   test_repos/{case}/*.py  ──►  Agente Claude  ──►  JSON    │
│                              (SUT)            decision     │
│                                               violations   │
│                                               reasoning    │
│                                    │                        │
│                                    ▼                        │
│                          golden_dataset.yaml               │
│                          (expected_decision,               │
│                           expected_violations)              │
│                                    │                        │
│                     ┌──────────────┼──────────────┐        │
│                     ▼              ▼              ▼        │
│              ExactMatch       RuleBased       LLMJudge     │
│           (rule_ids found) (schema válido) (reasoning ok) │
└─────────────────────────────────────────────────────────────┘
```

**El SUT es el agente**, no los archivos estáticos. Medimos:
- **Calidad**: ¿detecta las violaciones correctas? → precision / recall / F1 vs golden
- **Performance**: TTFT, TTC, tokens consumidos, coste USD

## ¿Qué evaluamos aquí?

### El SUT (System Under Test) es el agente

A diferencia de un linter clásico que analiza código directamente, aquí **le pedimos a Claude** que revise un repositorio y nos diga si hay violaciones. El agente recibe:

1. Las **reglas SDLC** inyectadas en el prompt
2. El **contenido completo** de los archivos del repositorio

Y devuelve un JSON estructurado:
```json
{
  "decision": "NO-GO",
  "violations": [{"rule_id": "use_internal_http", "file": "api_client.py", ...}],
  "reasoning": "El repositorio usa requests en lugar de httpx_internal..."
}
```

### ¿Por qué evaluar un agente y no un linter?

- Los linters detectan patrones exactos. Un agente puede entender **contexto semántico**
- Podemos medir si el agente **se equivoca** (falsos positivos/negativos) con un golden dataset
- El mismo patrón de eval aplica a cualquier agente de decisión (no solo código)

### Los 3 evaluadores actúan sobre el output del agente

| Evaluador | Qué comprueba |
|-----------|---------------|
| **ExactMatch** | Para cada `expected_violation.rule_id`, ¿aparece en la lista del agente? |
| **RuleBased** | ¿El JSON del agente tiene schema válido? (decision, violations[], reasoning) |
| **LLMJudge** | ¿El campo `reasoning` justifica correctamente la decisión? |

In [ ]:
# Setup completo — imports, dotenv, API key, paths
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import anthropic
import yaml

try:
    from dotenv import load_dotenv
    load_dotenv()
    print("python-dotenv loaded")
except ImportError:
    print("python-dotenv not installed — using environment variables directly")

# Verify API key
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    api_key = input("Introduce tu ANTHROPIC_API_KEY: ").strip()
    os.environ["ANTHROPIC_API_KEY"] = api_key

print(f"API key configurada: {'*' * 8}{api_key[-4:]}")


def _find_repo_root() -> Path:
    """Walk up from cwd until config/rules.yaml is found."""
    for candidate in [Path().resolve(), *Path().resolve().parents]:
        if (candidate / "config" / "rules.yaml").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del repo — ejecuta desde dentro del repositorio")


REPO_ROOT = _find_repo_root()
RULES_FILE = REPO_ROOT / "config" / "rules.yaml"
PROMPT_FILE = REPO_ROOT / "labs" / "01_sdlc_gatekeeper" / "agent_prompt.md"
GOLDEN_FILE = REPO_ROOT / "examples" / "test_repos" / "golden_dataset.yaml"
TEST_REPOS_DIR = REPO_ROOT / "examples" / "test_repos"
EXAMPLES_DIR = REPO_ROOT / "examples"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic(api_key=api_key)

print(f"Repo root: {REPO_ROOT}")
print(f"Rules file:   {RULES_FILE.exists()}")
print(f"Prompt file:  {PROMPT_FILE.exists()}")
print(f"Golden file:  {GOLDEN_FILE.exists()}")

## Referencias pedagógicas — ejemplos de buen y mal código

Los directorios `examples/good_code/` y `examples/bad_code/` contienen ejemplos de referencia.
**No son los test cases del eval** — son material didáctico para entender qué busca el agente.

El eval usa `examples/test_repos/{with_violations,clean}/` comparado contra `golden_dataset.yaml`.

In [ ]:
# Mostrar examples/good_code/api_client.py — código que pasa todas las reglas
good_api = (EXAMPLES_DIR / "good_code" / "api_client.py").read_text(encoding="utf-8")
print("=== examples/good_code/api_client.py ===")
print(good_api)

In [ ]:
# Mostrar examples/bad_code/api_client_bad.py — código con violaciones conocidas
bad_api = (EXAMPLES_DIR / "bad_code" / "api_client_bad.py").read_text(encoding="utf-8")
print("=== examples/bad_code/api_client_bad.py ===")
print(bad_api)

## Construcción del prompt del agente

El agente recibe un prompt con dos secciones variables:

- `{rules}`: las reglas SDLC del `config/rules.yaml` serializadas como texto legible
- `{codebase}`: todos los archivos `.py` del repositorio de test, concatenados en bloques markdown

El template está en `labs/01_sdlc_gatekeeper/agent_prompt.md`.

### ¿Por qué inyectar las reglas en el prompt?

Esto hace el agente **configurable sin reentrenar**. Si añadimos una regla nueva al YAML, el agente la respeta automáticamente en la siguiente llamada. Las reglas son el "sistema operativo" del agente.

In [ ]:
# Cargar rules.yaml y agent_prompt.md; construir el prompt del agente
with RULES_FILE.open("r", encoding="utf-8") as fh:
    all_rules = yaml.safe_load(fh)["rules"]

prompt_template = PROMPT_FILE.read_text(encoding="utf-8")


def serialize_rules(rules: list) -> str:
    """Format rules as readable text for the agent prompt."""
    lines = []
    for rule in rules:
        lines.append(f"- id: {rule['id']}")
        lines.append(f"  description: {rule['description']}")
        if "criteria" in rule:
            criteria_lines = rule["criteria"].strip().replace("\n", "\n      ")
            lines.append(f"  criteria: {criteria_lines}")
        if "pattern" in rule:
            lines.append(f"  pattern: {rule['pattern']} (forbidden)")
        if "forbidden_imports" in rule:
            lines.append(f"  forbidden_imports: {rule['forbidden_imports']}")
        if "max_args" in rule:
            lines.append(f"  max_args: {rule['max_args']}")
        lines.append("")
    return "\n".join(lines)


rules_text = serialize_rules(all_rules)
print(f"Reglas cargadas: {len(all_rules)}")
print(f"Prompt template (primeras 400 chars):")
print(prompt_template[:400])
print("...")
print(f"\nEjemplo de reglas serializadas (primeras 3):")
print("\n".join(rules_text.split("\n")[:15]))

## Test repos y golden dataset

Los **test repos** son repositorios sintéticos diseñados para tener violations conocidas:

- `examples/test_repos/with_violations/` — 3 archivos `.py` con violaciones deliberadas
- `examples/test_repos/clean/` — 3 archivos `.py` que cumplen todas las reglas

El **golden dataset** (`golden_dataset.yaml`) define:
- `expected_decision`: GO o NO-GO
- `expected_violations`: lista de `{rule_id, file}` que el agente debería detectar

Este golden es la **fuente de verdad** contra la que medimos precision/recall/F1.

In [ ]:
# Cargar golden_dataset.yaml y mostrar la estructura de test repos
with GOLDEN_FILE.open("r", encoding="utf-8") as fh:
    golden_data = yaml.safe_load(fh)

test_cases = golden_data["test_cases"]
print(f"Test cases en el golden: {len(test_cases)}")

for case in test_cases:
    print(f"\n--- case_id: {case['case_id']} ---")
    print(f"  repo_path:         {case['repo_path']}")
    print(f"  expected_decision: {case['expected_decision']}")
    print(f"  expected_violations ({len(case['expected_violations'])})")
    for v in case["expected_violations"]:
        print(f"    - rule_id={v['rule_id']}, file={v['file']}")

# Mostrar contenido del case with_violations
violations_dir = TEST_REPOS_DIR / "with_violations"
print(f"\n=== Archivos en test_repos/with_violations ===")
for py_file in sorted(violations_dir.glob("*.py")):
    content = py_file.read_text(encoding="utf-8")
    lines = len(content.splitlines())
    print(f"  {py_file.name}: {lines} líneas")

## Ejecutar el agente con streaming

Llamamos al agente con **streaming** para capturar métricas de performance en tiempo real:

- **TTFT** (Time To First Token): latencia hasta que el agente empieza a responder
- **TTC** (Time To Complete): latencia total de la llamada
- **OTPS** (Output Tokens Per Second): velocidad de generación
- **Tokens** y **cost_usd**: para análisis de coste por modelo

### ¿Por qué medir TTFT además de TTC?

En producción, TTFT determina la **latencia percibida por el usuario**. En un CI/CD gate, TTC determina cuánto tarda el pipeline. Son métricas complementarias.

In [ ]:
# Función run_agent con streaming + medición de performance
PRICING = {
    "claude-haiku-4-5-20251001": (0.80, 4.00),
    "claude-sonnet-4-6": (3.00, 15.00),
    "claude-opus-4-7": (15.00, 75.00),
}


def cost_usd(model: str, tin: int, tout: int) -> float:
    pin, pout = PRICING.get(model, (3.00, 15.00))
    return round((tin * pin + tout * pout) / 1_000_000, 6)


def extract_json(text: str) -> dict:
    """Robustly parse JSON from agent output, handling markdown fences."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        return {
            "decision": "NO-GO",
            "violations": [],
            "reasoning": f"PARSE_ERROR: {text[:200]}",
        }


def run_agent(prompt: str, model: str = MODEL) -> tuple:
    """Call agent with streaming; return (agent_output_dict, perf_metrics_dict)."""
    ttft = None
    start = time.perf_counter()

    with client.messages.stream(
        model=model,
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for event in stream:
            if ttft is None and event.type == "content_block_start":
                ttft = time.perf_counter() - start
        final_msg = stream.get_final_message()

    ttc = time.perf_counter() - start
    text = "".join(b.text for b in final_msg.content if hasattr(b, "text"))

    usage = final_msg.usage
    input_tokens = usage.input_tokens
    output_tokens = usage.output_tokens
    delta_s = max((ttc - (ttft or ttc)), 0.001)

    perf = {
        "ttft_ms": round((ttft or ttc) * 1000, 1),
        "ttc_ms": round(ttc * 1000, 1),
        "otps": round(output_tokens / delta_s, 1),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "cost_usd": cost_usd(model, input_tokens, output_tokens),
    }

    return extract_json(text), perf


print("run_agent() definido")

In [ ]:
# Llamar al agente sobre test_repos/with_violations; mostrar output + métricas
violations_case = next(c for c in test_cases if c["case_id"] == "with_violations")
repo_path = REPO_ROOT / violations_case["repo_path"]

# Serializar codebase
codebase_parts = []
for py_file in sorted(repo_path.glob("*.py")):
    content = py_file.read_text(encoding="utf-8")
    codebase_parts.append(f"## file: {py_file.name}\n```python\n{content}\n```")
codebase_text = "\n\n".join(codebase_parts)

# Construir prompt
prompt = prompt_template.replace("{rules}", rules_text).replace("{codebase}", codebase_text)

print(f"Prompt length: {len(prompt)} chars")
print("Llamando al agente...")

agent_output, perf = run_agent(prompt, MODEL)

print("\n=== Output del agente ===")
print(json.dumps(agent_output, indent=2))
print("\n=== Métricas de performance ===")
for k, v in perf.items():
    print(f"  {k}: {v}")

## Métricas de calidad — precision / recall / F1 vs golden

Comparamos las violaciones detectadas por el agente contra el golden dataset:

```
TP (True Positives)  = rule_ids en agente ∩ expected
FP (False Positives) = rule_ids en agente - expected  (falsa alarma)
FN (False Negatives) = expected - rule_ids en agente  (se le escapó)

Precision = TP / (TP + FP)   ← de lo que reportó, ¿cuánto era real?
Recall    = TP / (TP + FN)   ← de lo que había, ¿cuánto detectó?
F1        = 2 * P * R / (P + R)
```

En un gatekeeper, **recall** es más crítico: un FN (violación no detectada) es más peligroso que un FP (falsa alarma).

In [ ]:
# Función compute_quality; aplicarla al output del agente vs golden
def compute_quality(agent_output: dict, expected: dict) -> dict:
    """Compute TP/FP/FN and precision/recall/F1 vs expected violations."""
    agent_violations = agent_output.get("violations", [])
    expected_violations = expected.get("expected_violations", [])

    agent_rule_ids = {v["rule_id"] for v in agent_violations}
    expected_rule_ids = {v["rule_id"] for v in expected_violations}

    tp = len(agent_rule_ids & expected_rule_ids)
    fp = len(agent_rule_ids - expected_rule_ids)
    fn = len(expected_rule_ids - agent_rule_ids)

    precision = tp / (tp + fp) if (tp + fp) > 0 else (1.0 if tp == 0 and fn == 0 else 0.0)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    decision_match = agent_output.get("decision") == expected.get("expected_decision")

    return {
        "decision_match": decision_match,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    }


quality = compute_quality(agent_output, violations_case)

print("=== Métricas de calidad: case=with_violations ===")
print(f"  decision_match:  {quality['decision_match']}")
print(f"  true_positives:  {quality['true_positives']}")
print(f"  false_positives: {quality['false_positives']}")
print(f"  false_negatives: {quality['false_negatives']}")
print(f"  precision:       {quality['precision']}")
print(f"  recall:          {quality['recall']}")
print(f"  F1:              {quality['f1']}")

if quality["false_negatives"] > 0:
    detected = {v["rule_id"] for v in agent_output.get("violations", [])}
    expected_ids = {v["rule_id"] for v in violations_case["expected_violations"]}
    missed = expected_ids - detected
    print(f"\n  Violaciones no detectadas: {missed}")

## Los 3 evaluadores sobre el output del agente

Los evaluadores no analizan el código fuente directamente — analizan el **output JSON del agente**:

1. **ExactMatchEvaluator**: comprueba que cada `expected_violation.rule_id` del golden aparece en `agent_output.violations`
2. **RuleBasedEvaluator**: valida el schema del JSON (`decision` ∈ {GO, NO-GO}, `violations` es lista, cada violation tiene `rule_id` y `file`)
3. **LLMJudgeEvaluator**: llama a Claude para evaluar si el `reasoning` del agente justifica correctamente la decisión, dado TP/FP/FN

In [ ]:
# Definir los 3 evaluadores sobre el output del agente

class ExactMatchEvaluator:
    """Verifica que cada expected_violation.rule_id aparece en la lista del agente."""

    name = "exact_match"

    def evaluate(self, agent_output: dict, expected: dict) -> dict:
        agent_rule_ids = {v["rule_id"] for v in agent_output.get("violations", [])}
        expected_violations = expected.get("expected_violations", [])

        missing = [
            v["rule_id"]
            for v in expected_violations
            if v["rule_id"] not in agent_rule_ids
        ]
        passed = len(missing) == 0
        return {
            "name": self.name,
            "passed": passed,
            "score": 1.0 if passed else round(
                (len(expected_violations) - len(missing)) / max(len(expected_violations), 1), 4
            ),
            "evidence": (
                "Todos los rule_ids esperados están presentes"
                if passed
                else f"rule_ids no detectados: {missing}"
            ),
        }


class RuleBasedEvaluator:
    """Valida el schema JSON del output del agente."""

    name = "rule_based"

    def evaluate(self, agent_output: dict, _expected: dict) -> dict:
        errors = []
        decision = agent_output.get("decision")
        if decision not in ("GO", "NO-GO"):
            errors.append(f"decision debe ser GO o NO-GO, recibido: {decision!r}")

        violations = agent_output.get("violations")
        if not isinstance(violations, list):
            errors.append(f"violations debe ser lista, recibido: {type(violations).__name__}")
        else:
            for i, v in enumerate(violations):
                for field in ("rule_id", "file"):
                    if field not in v:
                        errors.append(f"violations[{i}] falta campo '{field}'")

        if "reasoning" not in agent_output:
            errors.append("falta campo 'reasoning'")

        passed = len(errors) == 0
        return {
            "name": self.name,
            "passed": passed,
            "score": 1.0 if passed else 0.0,
            "evidence": "Schema válido" if passed else "; ".join(errors),
        }


LLM_JUDGE_PROMPT = """You are evaluating the reasoning of an SDLC gatekeeper agent.

The agent reviewed code and produced this reasoning:
\"{agent_reasoning}\"

The agent's decision was: {agent_decision}
The expected decision was: {expected_decision}
The actual violations found vs expected: TP={tp}, FP={fp}, FN={fn}

Rate the reasoning from 0-10 on whether it correctly justifies the decision.
Respond ONLY with JSON: {{"score": <float>, "passed": <bool>, "reason": "..."}}
Pass threshold: score >= 6.0"""


class LLMJudgeEvaluator:
    """Usa Claude para evaluar la calidad del reasoning del agente."""

    name = "llm_judge"

    def evaluate(self, agent_output: dict, expected: dict, quality: dict) -> dict:
        prompt = LLM_JUDGE_PROMPT.format(
            agent_reasoning=agent_output.get("reasoning", ""),
            agent_decision=agent_output.get("decision", ""),
            expected_decision=expected.get("expected_decision", ""),
            tp=quality["true_positives"],
            fp=quality["false_positives"],
            fn=quality["false_negatives"],
        )
        try:
            msg = client.messages.create(
                model=MODEL,
                max_tokens=256,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = msg.content[0].text
            parsed = extract_json(raw)
            score = float(parsed.get("score", 0.0))
            passed = bool(parsed.get("passed", score >= 6.0))
            reason = str(parsed.get("reason", ""))
        except Exception as exc:
            return {"name": self.name, "passed": False, "score": 0.0,
                    "evidence": f"Judge error: {exc}"}

        return {
            "name": self.name,
            "passed": passed,
            "score": round(score, 2),
            "evidence": reason,
        }


exact_eval = ExactMatchEvaluator()
rule_eval = RuleBasedEvaluator()
llm_eval = LLMJudgeEvaluator()
print("Evaluadores definidos")

In [ ]:
# Ejecutar los 3 evaluadores sobre el output del agente (case: with_violations)
exact_result = exact_eval.evaluate(agent_output, violations_case)
rule_result = rule_eval.evaluate(agent_output, violations_case)
llm_result = llm_eval.evaluate(agent_output, violations_case, quality)

print("=== Resultados de evaluadores: case=with_violations ===")
for result in [exact_result, rule_result, llm_result]:
    status = "PASS" if result["passed"] else "FAIL"
    print(f"\n[{status}] {result['name']} (score={result['score']})")
    print(f"  {result['evidence']}")

## Pipeline completo — ambos test cases agregados

Ejecutamos el eval sobre **todos los test cases** del golden dataset y agregamos resultados:

- `avg_f1 >= 0.7` Y `todos los decision_match == True` → `overall_pass = True`
- Reportamos total de tokens y coste acumulado para comparar modelos

Este es exactamente el criterio que usa el script CLI (`01_sdlc_gatekeeper.py`) y el GitHub Action.

In [ ]:
# Loop sobre todos los test cases del golden dataset
all_results = []

for case in test_cases:
    print(f"\nEjecutando case: {case['case_id']} ...")
    repo_p = REPO_ROOT / case["repo_path"]

    # Serializar codebase
    parts = []
    for py_file in sorted(repo_p.glob("*.py")):
        content = py_file.read_text(encoding="utf-8")
        parts.append(f"## file: {py_file.name}\n```python\n{content}\n```")
    codebase = "\n\n".join(parts)

    prompt_case = prompt_template.replace("{rules}", rules_text).replace("{codebase}", codebase)

    agent_out, perf_metrics = run_agent(prompt_case, MODEL)
    q = compute_quality(agent_out, case)

    e_result = exact_eval.evaluate(agent_out, case)
    r_result = rule_eval.evaluate(agent_out, case)
    l_result = llm_eval.evaluate(agent_out, case, q)

    case_result = {
        "case_id": case["case_id"],
        "agent_output": agent_out,
        "quality": q,
        "performance": perf_metrics,
        "evaluators": [e_result, r_result, l_result],
    }
    all_results.append(case_result)

    print(f"  decision_match={q['decision_match']}  F1={q['f1']}  "
          f"ttft={perf_metrics['ttft_ms']}ms  cost=${perf_metrics['cost_usd']}")

# Tabla agregada
print("\n=== Tabla de resultados ===")
print(f"{'Case':<20} {'Decision':<8} {'Match':<7} {'P':<6} {'R':<6} {'F1':<6} {'Cost':<10}")
print("-" * 65)
for r in all_results:
    q = r["quality"]
    p = r["performance"]
    decision = r["agent_output"].get("decision", "?")
    print(f"{r['case_id']:<20} {decision:<8} {str(q['decision_match']):<7} "
          f"{q['precision']:<6} {q['recall']:<6} {q['f1']:<6} ${p['cost_usd']}")

In [ ]:
# Calcular aggregate + exportar JSON final a results/
n = len(all_results)
avg_precision = round(sum(r["quality"]["precision"] for r in all_results) / n, 4)
avg_recall = round(sum(r["quality"]["recall"] for r in all_results) / n, 4)
avg_f1 = round(sum(r["quality"]["f1"] for r in all_results) / n, 4)
all_decision_match = all(r["quality"]["decision_match"] for r in all_results)
overall_pass = avg_f1 >= 0.7 and all_decision_match

aggregate = {
    "overall_pass": overall_pass,
    "avg_precision": avg_precision,
    "avg_recall": avg_recall,
    "avg_f1": avg_f1,
    "total_input_tokens": sum(r["performance"]["input_tokens"] for r in all_results),
    "total_output_tokens": sum(r["performance"]["output_tokens"] for r in all_results),
    "total_cost_usd": round(sum(r["performance"]["cost_usd"] for r in all_results), 6),
    "total_ttft_ms": round(sum(r["performance"]["ttft_ms"] for r in all_results), 1),
}

report = {
    "run_id": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "lab": "lab01",
    "model": MODEL,
    "test_cases": all_results,
    "aggregate": aggregate,
}

timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M")
model_slug = MODEL.replace("/", "-")
output_path = RESULTS_DIR / f"{timestamp}_lab01_{model_slug}.json"
output_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(f"Resultado exportado: {output_path}")
print(f"\n=== Aggregate ===")
for k, v in aggregate.items():
    print(f"  {k}: {v}")
print(f"\noverall_pass = {overall_pass} (avg_f1={avg_f1} >= 0.7 AND all_decision_match={all_decision_match})")

## Reflexión — ¿qué modelo es el mejor gatekeeper?

Ejecuta este lab cambiando `MODEL` en la celda de setup para comparar:

| Modelo | TTFT típico | Recall esperado | Coste por run |
|--------|-------------|-----------------|---------------|
| `claude-haiku-4-5-20251001` | ~300ms | ~70-80% | ~$0.002 |
| `claude-sonnet-4-6` | ~500ms | ~85-95% | ~$0.02 |
| `claude-opus-4-7` | ~1-2s | ~95-99% | ~$0.10 |

### El dilema precision vs recall en un gate

- **Haiku** es rápido y barato — ideal para gates frecuentes (cada push)
- **Sonnet** es el punto de equilibrio — buena recall sin coste prohibitivo
- **Opus** tiene la mayor recall — adecuado para gates pre-release críticos

### La clave del eval

Sin un golden dataset no podríamos medir estos trade-offs. El eval transforma una pregunta cualitativa ("¿funciona bien el agente?") en una métrica objetiva (F1 = 0.87). Eso es lo que hace posible la optimización sistemática.